# SFT on StrategyQA + MMLU — Qwen2.5 1.5B via Unsloth (T4 16 GB)

**Model:** `Qwen/Qwen2.5-1.5B-Instruct`  
**Dataset:** `strategyqa` + `cais/mmlu` (~15 k samples)  
**Backend:** Unsloth + TRL `SFTTrainer` + sequence packing  
**Hardware target:** NVIDIA Tesla T4 — Compute Capability 7.5 (Turing, 16 GB VRAM)  
**Precision:** FP16 only (`fp16=True` / `bf16=False`) — Turing lacks BF16 registers  
**LoRA rank:** r = 16, targeting all linear modules  
**Gradient checkpointing:** Unsloth native (zero overhead variant)  
**Logging:** W&B — every step; sample outputs every 20 batches

---
### Why Unsloth over vanilla HF on a T4?
| Feature | Standard HF | Unsloth on T4 |
|---|---|---|
| Flash Attention | ❌ (needs CC ≥ 8.0) | ✅ via xformers + custom Triton kernels |
| Kernel-fused RoPE / CE loss | ❌ huge intermediate tensors | ✅ fused, never materialised in VRAM |
| Gradient checkpointing overhead | ~20% compute penalty | ~0% (Unsloth recompute graph) |
| BF16 | falls back to FP32 on T4 | n/a — Unsloth uses FP16 safely |
| LoRA adapter injection | manual peft | automatic + validated |
| Sequence packing | manual collator | `SFTTrainer(packing=True)` |

## Cell 1 — Install Unsloth & dependencies

Unsloth is pinned to a specific torch + CUDA combination for the T4 (CUDA 11.8, torch 2.4).  
The `--no-deps` flag on the second line prevents pip from accidentally downgrading torch.

In [1]:
# Pin the EXACT versions used by the working notebook.
# The working notebook: trl=0.24.0, transformers=5.5.0, unsloth-2026.6.1
# WHY: trl>=0.25 + transformers>=5.6 re-introduced the PicklingError with Unsloth.
# --no-deps on trl/peft/accelerate prevents pip from pulling newer transformers.
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-jufaxy4p/unsloth_624bfbf89fcd4a8cb8ad357692cea3aa
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-jufaxy4p/unsloth_624bfbf89fcd4a8cb8ad357692cea3aa
  Resolved https://github.com/unslothai/unsloth.git to commit 17e9714a98a361385511ae2a821899e934638876
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.4 MB/s eta 0:00:00
   ━━━

## Cell 2 — Imports

In [2]:
import shutil, os
cache_path = "/kaggle/working/unsloth_compiled_cache"
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Cache cleared.")

In [3]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

In [4]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import os, json, math, random, re, time
from dataclasses import dataclass, field
from typing import Dict, List, Optional
import numpy as np
import torch
# ── Unsloth MUST be imported before transformers ──────────────────────────────
# Unsloth patches torch ops in-place; importing it first guarantees the fused
# kernels (RoPE, cross-entropy, attention) are in effect for every subsequent
# model load.
from unsloth import FastLanguageModel
import transformers
from transformers import TrainingArguments, TrainerCallback
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
print("Torch        :", torch.__version__)
print("Transformers :", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    cc  = f"{dev.major}.{dev.minor}"
    print(f"GPU          : {dev.name}")
    print(f"VRAM         : {dev.total_memory / 1e9:.1f} GB")
    print(f"Compute Cap  : {cc}")
    if float(cc) < 7.5:
        raise RuntimeError("This notebook targets CC ≥ 7.5 (Turing / T4).")
    if float(cc) >= 8.0:
        print("ℹ️  CC ≥ 8.0 detected — you could use bf16, but fp16 is kept for T4 compat.")
from huggingface_hub import login

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch        : 2.10.0+cu128
Transformers : 5.5.0
CUDA available: True
GPU          : Tesla T4
VRAM         : 15.6 GB
Compute Cap  : 7.5


## Cell 3 — W&B login

## Cell 4 — Config

Key differences from the original notebook:
- `lora_r = 16` (was 64) — sufficient for a smaller dataset and to reduce overfitting
- `lora_target_modules`: all linear layers (Q, K, V, O, gate, up, down projections)
- `fp16 = True`, `bf16 = False` — mandatory on Turing silicon
- `use_gradient_checkpointing = "unsloth"` — zero-overhead variant
- `packing = True` — sequence packing via SFTTrainer; eliminates wasted pad compute
- `per_device_batch = 4, grad_accum = 8` → effective batch = 32 (fits 16 GB at max_seq_len=512)

In [5]:
# ── 4. Config ─────────────────────────────────────────────────────────────────
CFG = dict(
    # Model
    model_name        = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_len       = 2048,             # must match packing window

    # LoRA
    lora_r            = 16,              # rank — sufficient for smaller dataset
    lora_alpha        = 8,              # keep alpha == r (scale factor = 1)
    lora_dropout      = 0.05,
    lora_target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        
    ],
    bias              = "none",
    use_rslora        = True,            # rsLoRA normalises adapters — recommended at r=64

    # Dataset
    dataset_name      = "strategyqa+mmlu", # Placeholder, actual loading is in Cell 7
    val_size          = 2000,
    seed              = 42,

    # Training
    output_dir        = "/kaggle/working/qwen_strategy_mmlu_sft",
    num_train_epochs  = 3,               # more epochs for smaller dataset
    per_device_batch  = 4,              # T4 16 GB @ seq_len=512 with packing
    grad_accum        = 4,              # effective batch = 32
    learning_rate     = 1e-4,           # slightly lower to prevent overfitting
    warmup_ratio      = 0.1,            # more warmup
    lr_scheduler_type = "cosine",
    weight_decay      = 0.05,           # stronger regularisation
    max_grad_norm     = 1.0,

    # Precision — CRITICAL for T4 (Turing CC 7.5)
    fp16              = True,            # ✅ Turing supports FP16 Tensor Cores
    bf16              = False,           # ❌ Turing has NO BF16 registers

    # Unsloth gradient checkpointing
    use_gradient_checkpointing = "unsloth",  # zero-overhead recompute graph

    # Sequence packing (eliminates padding waste across 395k short samples)
    packing           = True,

    # Logging
    logging_steps     = 10,
    save_steps        = 200,
    save_total_limit  = 2,
    eval_steps        = 100,
    dataloader_workers= 2,
    report_to         = "wandb",
    run_name          = "qwen-strategy-mmlu-sft",
    print_sample_every= 20,

    # Checkpoint selection
    load_best_model   = True,
    metric_for_best   = "eval_loss",
)

os.makedirs(CFG["output_dir"], exist_ok=True)
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

{
  "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
  "max_seq_len": "2048",
  "lora_r": "16",
  "lora_alpha": "8",
  "lora_dropout": "0.05",
  "lora_target_modules": "['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']",
  "bias": "none",
  "use_rslora": "True",
  "dataset_name": "strategyqa+mmlu",
  "val_size": "2000",
  "seed": "42",
  "output_dir": "/kaggle/working/qwen_strategy_mmlu_sft",
  "num_train_epochs": "3",
  "per_device_batch": "4",
  "grad_accum": "4",
  "learning_rate": "0.0001",
  "warmup_ratio": "0.1",
  "lr_scheduler_type": "cosine",
  "weight_decay": "0.05",
  "max_grad_norm": "1.0",
  "fp16": "True",
  "bf16": "False",
  "use_gradient_checkpointing": "unsloth",
  "packing": "True",
  "logging_steps": "10",
  "save_steps": "200",
  "save_total_limit": "2",
  "eval_steps": "100",
  "dataloader_workers": "2",
  "report_to": "wandb",
  "run_name": "qwen-strategy-mmlu-sft",
  "print_sample_every": "20",
  "load_best_model": "True",
  "metric_for_be

## Cell 5 — Load model & tokeniser via `FastLanguageModel`

Unsloth's `FastLanguageModel.from_pretrained` does **four things in one call**:
1. Downloads / loads the model weights
2. Quantises to 4-bit NF4 (via bitsandbytes) to halve the base-model VRAM footprint — **the 1.5B base takes ~0.8 GB instead of ~3 GB**
3. Replaces standard attention with xformers memory-efficient attention (Flash Attn 2 semantics, CC ≥ 7.5 compatible)
4. Rewrites RoPE, cross-entropy, and layer-norm ops with fused Triton kernels

Afterwards `get_peft_model` injects LoRA adapters and activates Unsloth's native gradient checkpointing.

In [6]:
# ── 5. Load base model + inject LoRA via Unsloth ─────────────────────────────

print("Loading model + tokeniser via FastLanguageModel…")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = CFG["model_name"],
    max_seq_length = CFG["max_seq_len"],
    dtype        = None,            # auto: fp16 on T4
    load_in_4bit = False,            # NF4 quantisation → halves VRAM for base weights
    # token = "hf_...",            # add HF token here if gated model
)

# Ensure pad token is set (Qwen uses <|endoftext|> as both)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("\nInjecting LoRA adapters…")
model = FastLanguageModel.get_peft_model(
    model,
    r                   = CFG["lora_r"],
    target_modules      = CFG["lora_target_modules"],
    lora_alpha          = CFG["lora_alpha"],
    lora_dropout        = CFG["lora_dropout"],
    bias                = CFG["bias"],
    use_rslora          = CFG["use_rslora"],
    # Unsloth native gradient checkpointing — zero compute overhead
    # (replaces model.gradient_checkpointing_enable() entirely)
    use_gradient_checkpointing = CFG["use_gradient_checkpointing"],
    random_state        = CFG["seed"],
)

# Print trainable parameters
total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params     : {total_p/1e6:.1f} M")
print(f"Trainable params : {trainable_p/1e6:.1f} M  ({100*trainable_p/total_p:.2f}%)")
print(f"\n✅ LoRA rank r={CFG['lora_r']} injected into: {CFG['lora_target_modules']}")
print(f"✅ Precision: fp16={CFG['fp16']}  bf16={CFG['bf16']}  (T4 safe)")
print(f"✅ Gradient checkpointing: {CFG['use_gradient_checkpointing']} (Unsloth native)")

Loading model + tokeniser via FastLanguageModel…
==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

...

unsloth/Qwen2.5-1.5B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.



Injecting LoRA adapters…


Unsloth 2026.6.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.



Total params     : 1562.2 M
Trainable params : 18.5 M  (1.18%)

✅ LoRA rank r=16 injected into: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
✅ Precision: fp16=True  bf16=False  (T4 safe)
✅ Gradient checkpointing: unsloth (Unsloth native)


## Cell 6 — Prompt template

We use an Alpaca-style template that wraps the StrategyQA and MMLU `query` field.
The template is also fed to `SFTTrainer` as a formatting function, which is the
interface `trl` expects when `packing=True`.

In [7]:
# ── 6. Prompt template ────────────────────────────────────────────────────────
# Alpaca-style — same format as the original notebook so training signal is
# directly comparable.  The EOS token is appended so the model learns to stop.

ALPACA_TEMPLATE = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:"
)

def formatting_func(example) -> List[str]:
    """Called by SFTTrainer for each batch element.

    Returns a list of fully-formatted strings (prompt + response + EOS).
    SFTTrainer with packing=True concatenates these into 512-token blocks.
    """
    outputs = []
    # Handle both single examples and batched dicts
    queries    = example["query"]    if isinstance(example["query"],    list) else [example["query"]]
    responses  = example["response"] if isinstance(example["response"], list) else [example["response"]]
    for q, r in zip(queries, responses):
        prompt = ALPACA_TEMPLATE.format(instruction=q.strip())
        full   = prompt + r.strip() + tokenizer.eos_token
        outputs.append(full)
    return outputs

# Sanity check
print(formatting_func({"query": "What is 2+2?", "response": "The answer is 4."})[0])

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is 2+2?

### Response:The answer is 4.<|im_end|>


## Cell 7 — Load dataset & train/val split

**CHANGED:** This cell now loads `strategyqa` and `cais/mmlu`, maps them to the required format, and samples a total of 15,000 examples.

In [8]:
# ── 7. Dataset ────────────────────────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets
import random

print("Loading StrategyQA from metaeval/strategy-qa...")
strategy_ds = load_dataset("metaeval/strategy-qa", split="train")

def strategy_map(example):
    return {
        "query": example["question"],
        "response": str(example["answer"])  # ✅ Convert bool to string
    }

strategy_ds = strategy_ds.map(strategy_map, remove_columns=strategy_ds.column_names)
print(f"StrategyQA size: {len(strategy_ds)}")

print("\nLoading MMLU (auxiliary_train split)...")
try:
    mmlu_ds = load_dataset("cais/mmlu", "all", split="auxiliary_train")
except Exception as e:
    print(f"'auxiliary_train' not found, falling back to 'test' split. Error: {e}")
    mmlu_ds = load_dataset("cais/mmlu", "all", split="test")

def mmlu_to_qa(example):
    options = "\n".join([f"{chr(65+i)}. {opt}" for i, opt in enumerate(example["choices"])])
    query = f"{example['question']}\n\nOptions:\n{options}\n\nWhich option is correct? Answer with the letter only."
    answer_letter = chr(65 + example["answer"])
    return {"query": query, "response": answer_letter}  # string

mmlu_ds = mmlu_ds.map(mmlu_to_qa, remove_columns=mmlu_ds.column_names)
print(f"MMLU size: {len(mmlu_ds)}")

# Combine and sample to ~15,000 examples
print("\nCombining and sampling datasets...")
combined_ds = concatenate_datasets([strategy_ds, mmlu_ds])  # ✅ Now both have string 'response'
combined_ds = combined_ds.shuffle(seed=CFG["seed"])

# Sample 15,000 examples total (or use all if less)
total_samples = min(15000, len(combined_ds))
combined_ds = combined_ds.select(range(total_samples))
print(f"Total combined size after sampling: {len(combined_ds)}")

# Split into train/val
val_size = CFG["val_size"]
train_raw = combined_ds.select(range(len(combined_ds) - val_size))
val_raw = combined_ds.select(range(len(combined_ds) - val_size, len(combined_ds)))
print(f"Train : {len(train_raw):,}  |  Val : {len(val_raw):,}")

# Leak guard
train_q = set(train_raw["query"])
val_q   = set(val_raw["query"])
overlap = train_q & val_q
print(f"Overlap between splits: {len(overlap)}")

Loading StrategyQA from metaeval/strategy-qa...


strategyQA_train.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2290 [00:00<?, ? examples/s]

StrategyQA size: 2290

Loading MMLU (auxiliary_train split)...


...

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

Map:   0%|          | 0/99842 [00:00<?, ? examples/s]

MMLU size: 99842

Combining and sampling datasets...
Total combined size after sampling: 15000
Train : 13,000  |  Val : 2,000
Overlap between splits: 4


## Cell 8 — Rich metrics callback

Kept from the original notebook with minor adaptations:
- `_generate_samples` now calls `FastLanguageModel.for_inference(model)` / `FastLanguageModel.for_training(model)` to switch between Unsloth inference and training modes cleanly.

In [9]:
# RichMetricsCallback removed -- it was causing issues with newer transformers.
# Logging is handled by the built-in trainer logging_steps parameter.
print("Using built-in logging (no custom callbacks).")

Using built-in logging (no custom callbacks).


## Cell 9 — Training arguments

Key flags:
- `fp16=True`, `bf16=False` — mandatory for T4
- `optim="adamw_8bit"` — 8-bit Adam from bitsandbytes; cuts optimiser state VRAM in half vs FP32 Adam with no accuracy loss
- `gradient_checkpointing=True` is still set here but Unsloth overrides the internal implementation with its zero-overhead variant (activated when `use_gradient_checkpointing="unsloth"` was passed to `get_peft_model`)

In [10]:
# TrainingArguments are now defined INLINE inside SFTTrainer (see next cell)
# This matches the working notebook architecture exactly.
# Key difference: SFTConfig is passed as args=SFTConfig(...) directly,
# which avoids the PicklingError because trl never patches SFTConfig
# when it is created AFTER FastLanguageModel has already patched trl.
print("TrainingArguments will be defined inline in the SFTTrainer cell.")

TrainingArguments will be defined inline in the SFTTrainer cell.


## Cell 10 — SFTTrainer with sequence packing

**Why sequence packing matters here:**  
MetaMathQA samples average ~120–180 tokens. With a 512-token context window, standard padding wastes 60–70% of every forward pass on zero-valued pad tokens. `packing=True` tells `SFTTrainer` to concatenate multiple samples (separated by EOS) into a single 512-token chunk, bringing GPU utilisation close to 100%.

Combined with Unsloth's kernel fusion, this is the second biggest speed multiplier after the LoRA VRAM savings.

In [11]:
import threading, time

def keep_alive():
    while True:
        time.sleep(30)
        # Touch a file to signal activity
        with open("/kaggle/working/.keepalive", "w") as f:
            f.write(str(time.time()))

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("Keep-alive thread running.")

Keep-alive thread running.


In [12]:
# Login to Hugging Face Hub (uses secret HF_TOKEN stored in Kaggle)
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to Hugging Face Hub")

# Your HF repo where checkpoints will be pushed automatically
HF_REPO = "Suryansh7123/qwen2.5_lora_r16_finetune_STRATEGY"  # <-- change this
print(f"Checkpoints will push to: {HF_REPO}")

Logged in to Hugging Face Hub
Checkpoints will push to: Suryansh7123/qwen2.5_lora_r16_finetune_STRATEGY


*(HOTFIX cell retired — no longer needed)*

In [13]:
# (HOTFIX cell no longer needed — PicklingError is fixed by using
#  SFTConfig inline inside SFTTrainer, not a separate TrainingArguments)
print("No hotfix needed with the new SFTConfig-inline architecture.")

No hotfix needed with the new SFTConfig-inline architecture.


## Pre-flight Smoke Test

Run this cell **before** `trainer.train()`.  
It exercises the full pipeline (train -> eval -> checkpoint-save) on 16 examples (~30 s).  
The HOTFIX cell above must run first (it patches `torch.save`).
If all checks pass the real training run is safe to launch.

In [14]:
# =========================================================================
# PRE-FLIGHT SMOKE TEST
# Exercises train -> eval -> checkpoint-save on 16 examples (~30 s).
# Uses SFTConfig inline (same pattern as the working notebook).
# No PicklingError expected with this architecture.
# =========================================================================
import os, gc, shutil, traceback

PASS = True
def _ok(m):   print("  [OK]  " + str(m))
def _fail(m, e=None):
    global PASS; PASS = False
    print("  [FAIL] " + str(m))
    if e: traceback.print_exc()
def _sec(t): print("\n" + "-"*60 + "\n  " + t + "\n" + "-"*60)

print("=" * 65)
print("  PRE-FLIGHT SMOKE TEST")
print("=" * 65)

# -- 0. Version check -- this is why the working notebook works! -----------
_sec("0 . Library versions (must match working notebook)")
import transformers, trl, unsloth
_tv = transformers.__version__
_trlv = trl.__version__
print(f"  transformers : {_tv}")
print(f"  trl          : {_trlv}")
if _tv not in ("5.5.0",) and not _tv.startswith("5."):
    _fail(f"transformers {_tv} -- expected ~5.5.0 (working version)")
else:
    _ok(f"transformers {_tv}")
if _trlv not in ("0.24.0",):
    _fail(f"trl {_trlv} -- expected 0.24.0 (working version)")
else:
    _ok(f"trl {_trlv}")

# -- 1. GPU check ----------------------------------------------------------
_sec("1 . GPU / VRAM")
if torch.cuda.is_available():
    _dev = torch.cuda.get_device_properties(0)
    _cc  = float(f"{_dev.major}.{_dev.minor}")
    _gb  = _dev.total_memory / 1e9
    print(f"  GPU  : {_dev.name}  |  CC {_cc}  |  {_gb:.1f} GB")
    if _cc < 7.5:  _fail(f"CC {_cc} < 7.5")
    else:          _ok(f"CC {_cc} >= 7.5")
    if _gb < 14.0: _fail(f"Only {_gb:.1f} GB VRAM")
    else:          _ok(f"{_gb:.1f} GB VRAM")
else:
    _fail("CUDA not available")

# -- 2. Tiny dataset -------------------------------------------------------
_sec("2 . Tiny dataset (16 train / 4 val)")
try:
    tiny_train = train_raw.select(range(16))
    tiny_val   = val_raw.select(range(4))
    _ok(f"Sliced  train={len(tiny_train)}  val={len(tiny_val)}")
except Exception as e:
    _fail("Dataset slice failed", e)

# -- 3. formatting_func ----------------------------------------------------
_sec("3 . formatting_func sanity")
try:
    _s = formatting_func({"query": "What is 2+2?", "response": "4"})
    assert len(_s) == 1 and tokenizer.eos_token in _s[0]
    _ok("formatting_func returns valid string with EOS token")
except Exception as e:
    _fail("formatting_func check failed", e)

# -- 4. Smoke SFTTrainer with inline SFTConfig (working notebook pattern) --
_sec("4 . Smoke SFTTrainer (inline SFTConfig)")
SMOKE_DIR = "/kaggle/working/_smoke_ckpt"
os.makedirs(SMOKE_DIR, exist_ok=True)
smoke_trainer = None
if PASS:
    try:
        smoke_trainer = SFTTrainer(
            model            = model,
            tokenizer        = tokenizer,
            train_dataset    = tiny_train,
            eval_dataset     = tiny_val,
            formatting_func  = formatting_func,
            args = SFTConfig(
                output_dir                    = SMOKE_DIR,
                max_seq_length                = CFG["max_seq_len"],
                packing                       = False,
                dataset_num_proc              = 1,
                num_train_epochs              = 1,
                max_steps                     = 4,
                per_device_train_batch_size   = CFG["per_device_batch"],
                gradient_accumulation_steps   = 2,
                learning_rate                 = CFG["learning_rate"],
                weight_decay                  = CFG["weight_decay"],
                warmup_steps                  = 0,
                max_grad_norm                 = CFG["max_grad_norm"],
                fp16                          = CFG["fp16"],
                bf16                          = CFG["bf16"],
                optim                         = "adamw_8bit",
                gradient_checkpointing        = True,
                gradient_checkpointing_kwargs = {"use_reentrant": False},
                logging_steps                 = 1,
                save_steps                    = 2,
                save_total_limit              = 1,
                eval_strategy                 = "steps",
                eval_steps                    = 2,
                per_device_eval_batch_size    = CFG["per_device_batch"],
                load_best_model_at_end        = False,
                report_to                     = "none",
                seed                          = CFG["seed"],
                remove_unused_columns         = False,
            ),
        )
        _ok("Smoke SFTTrainer initialised")
    except Exception as e:
        _fail("SFTTrainer init failed", e)

# -- 5. Smoke train() (checkpoint-save path) -------------------------------
_sec("5 . smoke_trainer.train()  [4 steps + checkpoint save]")
if PASS and smoke_trainer is not None:
    try:
        gc.collect(); torch.cuda.empty_cache()
        _st = smoke_trainer.train()
        _ok(f"train() finished -- loss={_st.training_loss:.4f}")
    except Exception as e:
        _fail(f"trainer.train() raised {type(e).__name__}: {e}", e)

# -- 6. Checkpoint check ---------------------------------------------------
_sec("6 . Checkpoint file check")
if PASS:
    _ckpts = [d for d in os.listdir(SMOKE_DIR) if d.startswith("checkpoint")]
    if _ckpts:
        _cp    = os.path.join(SMOKE_DIR, _ckpts[0])
        _files = os.listdir(_cp)
        _has_st = any(f.endswith(".safetensors") for f in _files)
        if _has_st: _ok("checkpoint contains .safetensors")
        else:       _fail(f"No .safetensors -- files: {_files[:6]}")
    else:
        _fail(f"No checkpoint in {SMOKE_DIR}")

# -- 7. Inference ----------------------------------------------------------
_sec("7 . Inference sanity")
if PASS:
    try:
        FastLanguageModel.for_inference(model)
        _enc = tokenizer(
            ALPACA_TEMPLATE.format(instruction="What is 2+2?"),
            return_tensors="pt", truncation=True, max_length=128
        ).to(model.device)
        with torch.no_grad():
            _out = model.generate(**_enc, max_new_tokens=20, do_sample=False,
                                  pad_token_id=tokenizer.pad_token_id,
                                  eos_token_id=tokenizer.eos_token_id, use_cache=True)
        _resp = tokenizer.decode(_out[0][_enc.input_ids.shape[1]:], skip_special_tokens=True)
        _ok(f"Inference OK -> '{_resp[:60]}'")
        FastLanguageModel.for_training(model)
        _ok("Restored model to training mode")
    except Exception as e:
        _fail(f"Inference failed: {e}", e)
        try: FastLanguageModel.for_training(model)
        except: pass

# -- 8. VRAM ---------------------------------------------------------------
_sec("8 . GPU memory")
gc.collect(); torch.cuda.empty_cache()
_peak = torch.cuda.max_memory_reserved() / 1e9
print(f"  Peak VRAM : {_peak:.2f} GB")
if _peak > 15.5: _fail(f"Peak {_peak:.2f} GB > 15.5 GB")
else:            _ok(f"Peak {_peak:.2f} GB -- safe")

# -- 9. Cleanup ------------------------------------------------------------
_sec("9 . Cleanup")
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
del smoke_trainer, tiny_train, tiny_val
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
_ok("Done")

print()
print("=" * 65)
if PASS:
    print("  [PASS] PRE-FLIGHT PASSED -- safe to run trainer.train()")
else:
    print("  [FAIL] PRE-FLIGHT FAILED -- fix errors above")
    raise RuntimeError("Pre-flight failed.")
print("=" * 65)

  PRE-FLIGHT SMOKE TEST

------------------------------------------------------------
  0 . Library versions (must match working notebook)
------------------------------------------------------------
  transformers : 5.5.0
  trl          : 0.24.0
  [OK]  transformers 5.5.0
  [OK]  trl 0.24.0

------------------------------------------------------------
  1 . GPU / VRAM
------------------------------------------------------------
  GPU  : Tesla T4  |  CC 7.5  |  15.6 GB
  [OK]  CC 7.5 >= 7.5
  [OK]  15.6 GB VRAM

------------------------------------------------------------
  2 . Tiny dataset (16 train / 4 val)
------------------------------------------------------------
  [OK]  Sliced  train=16  val=4

------------------------------------------------------------
  3 . formatting_func sanity
------------------------------------------------------------
  [OK]  formatting_func returns valid string with EOS token

------------------------------------------------------------
  4 . Smoke SFTT

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/16 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/4 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
  [OK]  Smoke SFTTrainer initialised

------------------------------------------------------------
  5 . smoke_trainer.train()  [4 steps + checkpoint save]
------------------------------------------------------------


...

Step,Training Loss,Validation Loss
2,2.421470,2.404348
4,2.199447,2.364030


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_smoke_ckpt/checkpoint-2/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_smoke_ckpt/checkpoint-4/tokenizer_config.json.
Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [OK]  train() finished -- loss=2.2968

------------------------------------------------------------
  6 . Checkpoint file check
------------------------------------------------------------
  [OK]  checkpoint contains .safetensors

------------------------------------------------------------
  7 . Inference sanity
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  [OK]  Inference OK -> ' The answer to "What is 2+2?" is 4. This is because when you'
  [OK]  Restored model to training mode

------------------------------------------------------------
  8 . GPU memory
------------------------------------------------------------
  Peak VRAM : 7.32 GB
  [OK]  Peak 7.32 GB -- safe

------------------------------------------------------------
  9 . Cleanup
------------------------------------------------------------
  [OK]  Done

  [PASS] PRE-FLIGHT PASSED -- safe to run trainer.train()


## Cell 11 — Train

In [15]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [16]:
# ── Guard: (re)create trainer if not already defined ─────────────────────────
# Run this cell if you get: NameError: name 'trainer' is not defined
# This happens when the SFTTrainer cell above crashed or was skipped.
import gc, torch
gc.collect(); torch.cuda.empty_cache()

if 'trainer' not in dir():
    print("trainer not found -- recreating...")
    trainer = SFTTrainer(
        model            = model,
        tokenizer        = tokenizer,
        train_dataset    = train_raw,
        eval_dataset     = val_raw,
        formatting_func  = formatting_func,

        args = SFTConfig(
            output_dir                   = CFG["output_dir"],
            max_seq_length               = CFG["max_seq_len"],
            packing                      = False,
            dataset_num_proc             = 1,

            num_train_epochs             = CFG["num_train_epochs"],
            per_device_train_batch_size  = CFG["per_device_batch"],
            gradient_accumulation_steps  = CFG["grad_accum"],
            learning_rate                = CFG["learning_rate"],
            weight_decay                 = CFG["weight_decay"],
            warmup_ratio                 = CFG["warmup_ratio"],
            lr_scheduler_type            = CFG["lr_scheduler_type"],
            max_grad_norm                = CFG["max_grad_norm"],

            fp16                         = CFG["fp16"],
            bf16                         = CFG["bf16"],

            optim                        = "adamw_8bit",
            gradient_checkpointing       = True,
            gradient_checkpointing_kwargs= {"use_reentrant": False},

            logging_steps                = CFG["logging_steps"],
            logging_first_step           = True,

            save_strategy                = "steps",
            save_steps                   = CFG["save_steps"],
            save_total_limit             = CFG["save_total_limit"],

            # -- Push to Hugging Face Hub at every checkpoint --
            push_to_hub                  = True,
            hub_model_id                 = HF_REPO,
            hub_strategy                 = "checkpoint",

            eval_strategy                = "steps",
            eval_steps                   = CFG["eval_steps"],
            per_device_eval_batch_size   = CFG["per_device_batch"] * 2,
            load_best_model_at_end       = CFG["load_best_model"],
            metric_for_best_model        = CFG["metric_for_best"],

            seed                         = CFG["seed"],
            remove_unused_columns        = False,
        ),
    )
    print("trainer created OK")
else:
    print("trainer already exists -- nothing to do")

print(f"train_dataset size : {len(trainer.train_dataset)}")
print(f"eval_dataset  size : {len(trainer.eval_dataset)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainer not found -- recreating...


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/13000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/2000 [00:00<?, ? examples/s]

trainer created OK
train_dataset size : 13000
eval_dataset  size : 2000


In [17]:
# ── 11. Train ─────────────────────────────────────────────────────────────────
print("Starting training…")
trainer_stats = trainer.train()   # Notice we removed resume_from_checkpoint!
print("Training complete.")
print(f"Peak VRAM : {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")
print(trainer_stats)

Starting training…


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 13,000 | Num Epochs = 3 | Total steps = 2,439
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,1.882813,1.888492
200,1.906480,1.872242
300,1.867761,1.863617
400,1.843869,1.859847
500,1.941970,1.854466
600,1.891094,1.847621
700,1.898417,1.843792
800,1.855514,1.840670
900,1.746009,1.839188
1000,1.816850,1.838116


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Training complete.
Peak VRAM : 15.27 GB
TrainOutput(global_step=2439, training_loss=1.7989242943939296, metrics={'train_runtime': 31505.2903, 'train_samples_per_second': 1.238, 'train_steps_per_second': 0.077, 'total_flos': 1.6681429363912704e+17, 'train_loss': 1.7989242943939296, 'epoch': 3.0})


## Cell 12 — Save (adapter-only *and* merged options)

Unsloth offers two export strategies:

| Strategy | Command | VRAM at save | Deployment |
|---|---|---|---|
| **Adapter only** (default) | `model.save_pretrained(path)` | ~50 MB | Dynamic: load base + adapters at inference. Portable, composable. |
| **Merged (fused) weights** | `model.save_pretrained_merged(path, tokenizer, save_method="merged_16bit")` | ~3 GB | Self-contained 1.5B checkpoint. Easier to serve via vLLM/Ollama. |

We save both. Adapt as needed.

In [18]:
# ── 12. Save — adapter-only ───────────────────────────────────────────────────
adapter_dir = os.path.join(CFG["output_dir"], "lora_adapter")
os.makedirs(adapter_dir, exist_ok=True)

model.save_pretrained(adapter_dir)          # saves adapter_config.json + adapter_model.bin
tokenizer.save_pretrained(adapter_dir)
print(f"LoRA adapter saved to → {adapter_dir}")
print("  Load at inference: FastLanguageModel.from_pretrained(adapter_dir) OR")
print("  PeftModel.from_pretrained(base_model, adapter_dir)")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen_strategy_mmlu_sft/lora_adapter/tokenizer_config.json.


LoRA adapter saved to → /kaggle/working/qwen_strategy_mmlu_sft/lora_adapter
  Load at inference: FastLanguageModel.from_pretrained(adapter_dir) OR
  PeftModel.from_pretrained(base_model, adapter_dir)


In [19]:
# ── 12b. Save — merged (optional) ────────────────────────────────────────────
# Fuses LoRA weights permanently into the base model and saves a standard
# HuggingFace checkpoint. Requires ~3 GB of free VRAM during the merge.
# Comment out if Kaggle disk quota is a concern.

# merged_dir = os.path.join(CFG["output_dir"], "merged_16bit")
# model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
# print(f"Merged 16-bit model saved to → {merged_dir}")

## Cell 13 — Post-training inference test

In [20]:
# ── 13. Inference test ────────────────────────────────────────────────────────
# Switch to Unsloth inference mode: enables KV-cache, disables training patches.
FastLanguageModel.for_inference(model)

test_questions = [
    "What is the sum of the first 10 natural numbers?",
    "If a train travels 60 km/h for 2.5 hours, how far does it travel?",
    "Solve for x: 3x + 7 = 22",
    "A rectangle has length 8 cm and width 5 cm. What is its area and perimeter?",
]

print("\n" + "═" * 70)
print("  POST-TRAINING INFERENCE TEST")
print("═" * 70)

for q in test_questions:
    prompt = ALPACA_TEMPLATE.format(instruction=q)
    enc    = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=CFG["max_seq_len"] // 2).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens  = 300,
            do_sample       = False,
            temperature     = 1.0,
            pad_token_id    = tokenizer.pad_token_id,
            eos_token_id    = tokenizer.eos_token_id,
            use_cache       = True,
        )
    response = tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 70)

# wandb.finish()
print("\nDone.")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



══════════════════════════════════════════════════════════════════════
  POST-TRAINING INFERENCE TEST
══════════════════════════════════════════════════════════════════════


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is the sum of the first 10 natural numbers?
A: 25
----------------------------------------------------------------------


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: If a train travels 60 km/h for 2.5 hours, how far does it travel?
A: 150km
----------------------------------------------------------------------


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Solve for x: 3x + 7 = 22
A: 15
----------------------------------------------------------------------

Q: A rectangle has length 8 cm and width 5 cm. What is its area and perimeter?
A: Area = 40 cm2; Perimeter = 36 cm Answer:B
----------------------------------------------------------------------

Done.
